In [16]:
import os
import ast
from typing import Dict

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from dotenv import load_dotenv

pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.multi_sparse', True)
pd.set_option('display.max_rows', 300)

load_dotenv(dotenv_path="/weka/home/adeluci2/scratch-mdredze1/adeluci2/Multi-Document-Summarization-Analysis/.env")

PROJECT_DIR = os.environ["MDS"]

In [17]:
metric_name_map = {
    "rouge_rouge1": "R-1",
    "rouge_rougeL": "R-L",
    "rouge_rougeLsum": "R-LSum",
    "moverscore": "Mover.",
    "unieval_relevance": "Rel. (UE)",
    "unieval_factuality": "Fact. (UE)", 
    "unieval_fluency": "Flue. (UE)",
    "unieval_coherence": "Coh. (UE)",
    "compression_sentence": "Comp.-Sent.",
    "compression_token": "Comp.-Word",
    "bertscore_recall": "BERTS. R",
    "bertscore_precision": "BERTS. P",
    "bertscore_f1": "BERTS. F1"
}

metric_to_category = {
    "rouge_rouge1": "Ground-Truth Similarity",
    "rouge_rougeL": "Ground-Truth Similarity",
    "rouge_rougeLsum": "Ground-Truth Similarity",
    "moverscore": "Ground-Truth Similarity",
    "unieval_relevance": "Ground-Truth Similarity",
    "unieval_factuality": "Factuality", 
    "unieval_fluency": "Quality",
    "unieval_coherence": "Quality",
    "compression_sentence": "Quality",
    "compression_token": "Quality",
    "bertscore_recall": "Ground-Truth Similarity",
    "bertscore_precision": "Ground-Truth Similarity",
    "bertscore_f1": "Ground-Truth Similarity"
}

dataset_name_map = {
    "convosumm": "Conv. Reddit",
    "convosumm/reddit": "Conv. Reddit",
    "multixscience": "XScience",
    "multinews_plus": "News+"
}

def get_model_type(raw_model_name: str) -> str:
    if "PRIMERA" in raw_model_name:
        return "PRIMERA"
    if "ICL" in raw_model_name:
        raw_model_name = raw_model_name.replace("ICL", "Llama 3.1")
    if "Llama 3.1" == raw_model_name:
        raw_model_name = "Llama 3.1 (0-shot)"
    else:
        raw_model_name = raw_model_name.split("-")[0]
    return raw_model_name


def get_train_domain(row: pd.Series) -> str:
    if row.model == "ground_truth":
        return dataset_name_map[row.dataset]
    if "icl" == row.model.lower():
        return "zero-shot"
    else:
        return dataset_name_map[row.model.split("-")[-1].lower()]


def format_entry(entry: str) -> Dict[str, float]:
    try:
        entry = ast.literal_eval(entry)
    except ValueError as err:
        print(f"Could not convert '{entry}' as literal")
        return pd.NA
    return entry
 

def load_aggregated_results(path: str) -> pd.DataFrame:
    # Load CSV. First two rows are header columns.
    df = pd.read_csv(path, header=[0,1])
    # Flatten the columns multi-index
    df.columns = pd.Index([
      f"{c[0]}_{c[1]}" if c[1]!="nan" else c[0] for c in df.columns
    ])
    metric_cols = [c for c in df.columns if c not in {"run_id", "model", "dataset", "split"}]
    
    # Convert the dictionaries from strings
    # and expand into individual columns
    # for col in metric_cols:
    #     df[col] = df[col].map(format_entry)
    #     for stat in ["mean", "std", "ci_std"]:
    #         df[f"{col}_{stat}"] = df[col].map(lambda x: x if pd.isna(x) else x[stat])
    #     df[f"{col}_ci_lower"] = df[col].map(lambda x: x if pd.isna(x) else x["ci"][0])
    #     df[f"{col}_ci_upper"] = df[col].map(lambda x: x if pd.isna(x) else x["ci"][1])

    df["dataset_test"] = df["dataset"].map(lambda x: dataset_name_map.get(x, x))
    df["dataset_train"] = df.apply(get_train_domain, axis="columns")
    df["model_type"] = df.model.map(get_model_type)
    df["in_domain"] = df.apply(lambda x: x.dataset_train==x.dataset_test, axis="columns")

    # --- 2. Data Transformation (from Wide to Long Format) ---
    # This part parses the string columns and reshapes the DataFrame.
    # The "long" format is the standard for plotting libraries like Seaborn.
    processed_records = []
    for index, row in df.iterrows():
        for col_name in metric_cols:
            try:
                # Safely evaluate the string to a dictionary
                metric_dict = ast.literal_eval(row[col_name])
                
                # Create a new record for the long-format DataFrame
                processed_records.append({
                    'model_type': row["model_type"],
                    'in_domain': row['in_domain'],
                    'dataset_test': row['dataset_test'],
                    'dataset_train': row["dataset_train"],
                    'metric': metric_name_map.get(col_name, col_name),
                    'metric_category': metric_to_category.get(col_name, "Other"),
                    'mean_score': metric_dict.get('mean', pd.NA),
                    'std_dev': metric_dict.get('std', pd.NA)
                })
            except (ValueError, SyntaxError):
                # Handle cases where the cell might be empty or malformed
                print(f"Could not parse row {index}, column {col_name}. Skipping.")
    df_long = pd.DataFrame(processed_records)

    return df, df_long


In [18]:
results_df, df_long = load_aggregated_results(f"{PROJECT_DIR}/results/experiments/aggregated_results.csv")

results_df

Could not parse row 40, column moverscore. Skipping.
Could not parse row 40, column bertscore_f1. Skipping.
Could not parse row 40, column bertscore_precision. Skipping.
Could not parse row 40, column bertscore_recall. Skipping.


,run_id,model,dataset,split,unieval_relevance,unieval_factuality,rouge_rouge1,rouge_rougeL,rouge_rougeLsum,moverscore,bertscore_f1,bertscore_precision,bertscore_recall,unieval_fluency,unieval_coherence,compression_sentence,compression_token,dataset_test,dataset_train,model_type,in_domain
0,1,ground_truth,convosumm/reddit,test,"{'mean': 0.9855425109640278, 'std': 0.03131239...","{'mean': 0.7127423688060528, 'std': 0.14718363...","{'mean': 1.0, 'std': 0.0, 'ci': (1.0, 1.0), 'c...","{'mean': 1.0, 'std': 0.0, 'ci': (1.0, 1.0), 'c...","{'mean': 1.0, 'std': 0.0, 'ci': (1.0, 1.0), 'c...","{'mean': 1.0, 'std': 0.0, 'ci': (1.0, 1.0), 'c...","{'mean': 0.9999999418258667, 'std': 1.38295929...","{'mean': 0.9999999418258667, 'std': 1.38295929...","{'mean': 0.9999999418258667, 'std': 1.38295929...","{'mean': 0.8962882354893531, 'std': 0.09558429...","{'mean': 0.729176020702826, 'std': 0.208525265...","{'mean': 0.10242261517227712, 'std': 0.0485502...","{'mean': 0.14928181353158385, 'std': 0.0799263...",Conv. Reddit,Conv. Reddit,ground_truth,True
1,2,allenai/PRIMERA-multixscience,convosumm/reddit,test,"{'mean': 0.5976378536643414, 'std': 0.27521109...","{'mean': 0.5085267988744268, 'std': 0.15112957...","{'mean': 0.21164999766803447, 'std': 0.0578874...","{'mean': 0.12443743614989292, 'std': 0.0324015...","{'mean': 0.12443743614989292, 'std': 0.0324015...","{'mean': 0.012527305853438704, 'std': 0.062514...","{'mean': -0.000358729700135882, 'std': 0.06785...","{'mean': -0.07653766959253698, 'std': 0.077606...","{'mean': 0.08139306873828173, 'std': 0.0705534...","{'mean': 0.5998391898054409, 'std': 0.17913964...","{'mean': 0.6405243598933591, 'std': 0.24893135...","{'mean': 0.23586857258883323, 'std': 0.1528116...","{'mean': 0.40346470180961297, 'std': 0.2337040...",Conv. Reddit,XScience,PRIMERA,False
2,3,PRIMERA-ConvoSumm,convosumm/reddit,test,"{'mean': 0.731714465971771, 'std': 0.254093978...","{'mean': 0.6483481734150773, 'std': 0.17214836...","{'mean': 0.36554962848924616, 'std': 0.0757385...","{'mean': 0.23304234760069606, 'std': 0.0554475...","{'mean': 0.23304234760069606, 'std': 0.0554475...","{'mean': 0.08901442503237685, 'std': 0.0763412...","{'mean': 0.21177772682905197, 'std': 0.0850327...","{'mean': 0.21977261752355845, 'std': 0.0931853...","{'mean': 0.1992487983852625, 'std': 0.08940725...","{'mean': 0.9152546237628535, 'std': 0.07876620...","{'mean': 0.7131287797315868, 'std': 0.23298629...","{'mean': 0.09956075453627942, 'std': 0.0396044...","{'mean': 0.13331725384642562, 'std': 0.0629207...",Conv. Reddit,Conv. Reddit,PRIMERA,True
3,4,PRIMERA-MultiNews_Plus,convosumm/reddit,test,"{'mean': 0.4663545987961307, 'std': 0.28373713...","{'mean': 0.9543511345592296, 'std': 0.08808043...","{'mean': 0.2052130924087042, 'std': 0.07015013...","{'mean': 0.11616032503358079, 'std': 0.0404427...","{'mean': 0.11616032503358079, 'std': 0.0404427...","{'mean': -0.011123763904697138, 'std': 0.07067...","{'mean': -0.03426252436544746, 'std': 0.106407...","{'mean': -0.11852404737100006, 'std': 0.120503...","{'mean': 0.06349818559701088, 'std': 0.1133708...","{'mean': 0.6916803453213957, 'std': 0.15235310...","{'mean': 0.9178626854662283, 'std': 0.17835100...","{'mean': 0.31899975411562603, 'std': 0.3375974...","{'mean': 0.3909981052937315, 'std': 0.33212922...",Conv. Reddit,News+,PRIMERA,False
4,5,SummN-MultiXScience,convosumm/reddit,test,"{'mean': 0.6236892103003036, 'std': 0.36479893...","{'mean': 0.2434601452116484, 'std': 0.25441539...","{'mean': 0.1119149225522074, 'std': 0.06172212...","{'mean': 0.09801577616872328, 'std': 0.0466282...","{'mean': 0.09801577616872328, 'std': 0.0466282...","{'mean': -0.15609310382533664, 'std': 0.076291...","{'mean': -0.24424022296071052, 'std': 0.127346...","{'mean': -0.3176295380759984, 'std': 0.1361264...","{'mean': -0.15838947359193117, 'std': 0.138660...","{'mean': 0.8053866430683386, 'std': 0.24364602...","{'mean': 0.575846709246058, 'std': 0.303174024...","{'mean': 0.20716038853030794, 'std': 0.1211811..

In [19]:
df_long

,model_type,in_domain,dataset_test,dataset_train,metric,metric_category,mean_score,std_dev
0,ground_truth,True,Conv. Reddit,Conv. Reddit,Rel. (UE),Ground-Truth Similarity,0.99,0.03
1,ground_truth,True,Conv. Reddit,Conv. Reddit,Fact. (UE),Factuality,0.71,0.15
2,ground_truth,True,Conv. Reddit,Conv. Reddit,R-1,Ground-Truth Similarity,1.00,0.00
3,ground_truth,True,Conv. Reddit,Conv. Reddit,R-L,Ground-Truth Similarity,1.00,0.00
4,ground_truth,True,Conv. Reddit,Conv. Reddit,R-LSum,Ground-Truth Similarity,1.00,0.00
...,...,...,...,...,...,...,...,...
537,Llama 3.1,True,News+,News+,BERTS. R,Ground-Truth Similarity,-0.03,0.04
538,Llama 3.1,True,News+,News+,Flue. (UE),Quality,0.95,0.03
539,Llama 3.1,True,News+,News+,Coh. (UE),Quality,0.77,0.20
540,Llama 3.1,True,News+,News+,Comp.-Sent.,Quality,0.68,0.27


# Q: Which model is the best in-domain?

In [23]:
in_domain = df_long[df_long.in_domain].copy()
in_domain.set_index(["metric_category", "model_type", "dataset_train"]).sort_index()

in_domain  dataset_test       metric  mean_score  std_dev
metric_category         model_type   dataset_train                                                           
Factuality              ConvoSumm    Conv. Reddit        True  Conv. Reddit   Fact. (UE)        0.64     0.17
                                     News+               True         News+   Fact. (UE)        0.14     0.10
                                     XScience            True      XScience   Fact. (UE)        0.60     0.19
                        Llama 3.1    Conv. Reddit        True  Conv. Reddit   Fact. (UE)        0.59     0.09
                                     News+               True         News+   Fact. (UE)        0.11     0.08
                                     XScience            True      XScience   Fact. (UE)        0.63     0.12
                        PRIMERA      Conv. Reddit        True  Conv. Reddit   Fact. (UE)        0.65     0.17
                                     News+               True         News+   Fact. (UE)        0.19     0.17
                                     XScience            True      XScience   Fact. (UE)        0.96     0.08
                        SummN        Conv. Reddit        True  Conv. Reddit   Fact. (UE)        0.65     0.19
                                     News+               True         News+   Fact. (UE)        0.14     0.12
                                     XScience            True      XScience   Fact. (UE)        0.50     0.25
                        ground_truth Conv. Reddit        True  Conv. Reddit   Fact. (UE)        0.71     0.15
                                     News+               True         News+   Fact. (UE)        0.70     0.12
                                     XScience            True      XScience   Fact. (UE)        0.55     0.17
Ground-Truth Similarity ConvoSumm    Conv. Reddit        True  Conv. Reddit    Rel. (UE)        0.70     0.26
                                     Conv. Reddit        True  Conv. Reddit          R-1        0.35     0.08
                                     Conv. Reddit        True  Conv. Reddit          R-L        0.24     0.06
                                     Conv. Reddit        True  Conv. Reddit       R-LSum        0.24     0.06
                                     Conv. Reddit        True  Conv. Reddit       Mover.        0.06     0.08
                                     Conv. Reddit        True  Conv. Reddit    BERTS. F1        0.21     0.09
                                     Conv. Reddit        True  Conv. Reddit     BERTS. P        0.25     0.10
                                     Conv. Reddit        True  Conv. Reddit     BERTS. R        0.16     0.09
                                     News+               True         News+    Rel. (UE)        0.79     0.18
                                     News+               True         News+          R-1        0.24     0.04
                                     News+               True         News+          R-L        0.12     0.02
                                     News+               True         News+       R-LSum        0.12     0.02
                                     News+               True         News+       Mover.       -0.08     0.04
                                     News+               True         News+    BERTS. F1       -0.02     0.04
                                     News+               True         News+     BERTS. P       -0.01     0.05
                                     News+               True         News+     BERTS. R       -0.03     0.05
                                     XScience            True      XScience    Rel. (UE)        0.50     0.26
                                     XScience            True      XScience          R-1        0.31     0.08
                                     XScience            True      XScience          R-L        0.18     0.04
                                     XScience            True      XScience       R-LSum    